## Step 1 — Install everything

Every package this notebook needs gets installed here, in one block, with **no imports
in between**. That's what lets the numpy/pandas ABI fix at the end of this cell take
effect without a kernel restart: as long as numpy/pandas/nibabel have never been imported
yet in this kernel's lifetime, the first import later on will simply pick up whatever
final, consistent versions are on disk after this cell finishes. The restart was only ever
needed because installs and imports used to be interleaved.

In [ ]:
# Kaggle already ships torch, numpy, scipy, huggingface_hub
!pip install -q brainles-preprocessing
!pip install -q monai safetensors nibabel neuroHarmonize
!pip install -q -U huggingface_hub

# Run this LAST, after the packages above, since some of their dependency resolution
# can silently drift numpy/pandas to mismatched versions. This re-pins them to a
# mutually consistent pair as the final word on what's installed.
!pip install -q --upgrade --force-reinstall --no-cache-dir numpy pandas

print("All installs complete.")

## Step 2 — First imports of the session (numpy/pandas/nibabel etc. have not been touched until now)

In [ ]:
import glob
import os
import tarfile

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm

print("numpy:", np.__version__, "| pandas:", pd.__version__)

## Hugging Face auth for BrainIAC

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF-TOKEN")
login(token=hf_token)

In [ ]:
from huggingface_hub import hf_hub_download

backbone_path = hf_hub_download(repo_id="eugenehp/brainiac", filename="backbone.safetensors")
idh_head_path = hf_hub_download(repo_id="eugenehp/brainiac", filename="idh.safetensors")

## Dataset paths

In [ ]:
brats_root = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
ucsf_root  = "/kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-v3-dataset"

brats_extract_dir = "/kaggle/working/brats2021"
ucsf_extract_dir  = "/kaggle/working/ucsf_pdgm"

print("BraTS root contents:", os.listdir(brats_root)[:5])
print("UCSF root contents:", os.listdir(ucsf_root)[:5])

## Extract BraTS2021 (packaged as `.tar` archives, ~13.4 GB uncompressed — fits Kaggle scratch disk)

In [ ]:
os.makedirs(brats_extract_dir, exist_ok=True)

tar_files = glob.glob(f"{brats_root}/*.tar")
print(f"Found {len(tar_files)} tar files:", [os.path.basename(t) for t in tar_files])

for tar_path in tar_files:
    print(f"Extracting {os.path.basename(tar_path)} ...")
    with tarfile.open(tar_path, "r") as tf:
        for member in tqdm(tf.getmembers(), desc=os.path.basename(tar_path)):
            tf.extract(member, brats_extract_dir)

n_extracted = sum(len(files) for _, _, files in os.walk(brats_extract_dir))
print("Done. Total files extracted:", n_extracted)

## Locate UCSF-PDGM files

Checks for several possible packaging formats (loose `.nii`/`.nii.gz`, `.tar`, `.tar.gz`/`.tgz`,
`.zip`) since different Kaggle re-uploads of the same dataset package it differently. If none
of those match, it prints a full directory tree and file-extension breakdown instead of just
failing — so if this still can't find anything, we'll know exactly what's actually there from
the printed output rather than guessing again.

In [ ]:
import zipfile
from collections import Counter


def describe_dir(path, max_depth=3, max_entries=8):
    """Print a top-level listing, a shallow tree, and file-extension counts for diagnosis."""
    print(f"--- Top-level listing of {path} ---")
    try:
        print(os.listdir(path))
    except Exception as e:
        print(f"Could not list {path}: {e}")
        return

    print(f"\n--- Tree (depth <= {max_depth}) ---")
    for root, dirs, files in os.walk(path):
        depth = root[len(path):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root) or root}/")
        for f in files[:max_entries]:
            print(f"{indent}  {f}")
        if len(files) > max_entries:
            print(f"{indent}  ... +{len(files) - max_entries} more")

    print("\n--- File extension counts (all files, any depth) ---")
    all_files = []
    for root, dirs, files in os.walk(path):
        all_files.extend(files)
    ext_counts = Counter(
        f[f.index('.'):] if '.' in f else '(no extension)'
        for f in all_files
    )
    for ext, count in ext_counts.most_common(15):
        print(f"  {ext}: {count}")


nii_gz     = glob.glob(f"{ucsf_root}/**/*.nii.gz", recursive=True)
nii_plain  = glob.glob(f"{ucsf_root}/**/*.nii", recursive=True)
tar_files  = (glob.glob(f"{ucsf_root}/**/*.tar", recursive=True)
              + glob.glob(f"{ucsf_root}/**/*.tar.gz", recursive=True)
              + glob.glob(f"{ucsf_root}/**/*.tgz", recursive=True))
zip_files  = glob.glob(f"{ucsf_root}/**/*.zip", recursive=True)

if nii_gz or nii_plain:
    n = len(nii_gz) + len(nii_plain)
    print(f"Found {n} loose NIfTI files directly under ucsf_root — no extraction needed.")
    ucsf_data_dir = ucsf_root

elif tar_files:
    print(f"Found {len(tar_files)} tar-family archives, extracting ...")
    os.makedirs(ucsf_extract_dir, exist_ok=True)
    for tar_path in tar_files:
        print(f"Extracting {os.path.basename(tar_path)} ...")
        with tarfile.open(tar_path, "r:*") as tf:  # r:* auto-detects gz/bz2/plain
            for member in tqdm(tf.getmembers(), desc=os.path.basename(tar_path)):
                tf.extract(member, ucsf_extract_dir)
    ucsf_data_dir = ucsf_extract_dir

elif zip_files:
    print(f"Found {len(zip_files)} zip archives, extracting ...")
    os.makedirs(ucsf_extract_dir, exist_ok=True)
    for zip_path in zip_files:
        print(f"Extracting {os.path.basename(zip_path)} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in tqdm(zf.namelist(), desc=os.path.basename(zip_path)):
                zf.extract(member, ucsf_extract_dir)
    ucsf_data_dir = ucsf_extract_dir

else:
    describe_dir(ucsf_root)
    raise FileNotFoundError(
        "No .nii/.nii.gz/.tar/.tar.gz/.tgz/.zip files found anywhere under ucsf_root. "
        "See the directory tree and extension counts printed above to figure out the "
        "actual packaging format, then adjust the glob patterns above accordingly."
    )

print("UCSF data directory set to:", ucsf_data_dir)

## Sanity checks — load one sample volume from each dataset, plus the clinical CSV

In [ ]:
# BraTS sample
sample_t1ce = glob.glob(f"{brats_extract_dir}/**/*_t1ce.nii.gz", recursive=True)[0]
img = nib.load(sample_t1ce)
print("BraTS T1CE shape/spacing:", img.shape, img.header.get_zooms())

In [ ]:
# UCSF sample — filename casing and compression vary across re-uploads, so try a few patterns
ucsf_t1c_patterns = [
    "*_T1c.nii.gz", "*_t1c.nii.gz",
    "*_T1c.nii",    "*_t1c.nii",
]
ucsf_t1c_candidates = []
for pattern in ucsf_t1c_patterns:
    ucsf_t1c_candidates += glob.glob(f"{ucsf_data_dir}/**/{pattern}", recursive=True)
    if ucsf_t1c_candidates:
        break

if not ucsf_t1c_candidates:
    describe_dir(ucsf_data_dir)
    raise FileNotFoundError(
        f"No T1c/t1c files found under ucsf_data_dir matching {ucsf_t1c_patterns}. "
        "See the directory tree/extensions printed above to find the real filename pattern."
    )

sample_t1c = ucsf_t1c_candidates[0]
img2 = nib.load(sample_t1c)
print("UCSF T1c shape/spacing:", img2.shape, img2.header.get_zooms())

In [ ]:
# UCSF clinical metadata
clinical_csv_candidates = glob.glob(f"{ucsf_data_dir}/**/*metadata*.csv", recursive=True)
if not clinical_csv_candidates:
    # fall back to any csv at all, in case this re-upload names it differently
    clinical_csv_candidates = glob.glob(f"{ucsf_data_dir}/**/*.csv", recursive=True)

if not clinical_csv_candidates:
    describe_dir(ucsf_data_dir)
    raise FileNotFoundError("No CSV files found under ucsf_data_dir — see the tree printed above.")

clinical_csv = clinical_csv_candidates[0]
print("Using clinical CSV:", clinical_csv)
df = pd.read_csv(clinical_csv)

print(df[["ID", "Sex", "Age at MRI", "IDH", "MGMT status"]].head())
print(df["IDH"].value_counts())